In [15]:
import numpy as np
from pathlib import Path
import logging
import sys
from datetime import datetime
import json

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Import SecureEmbedding
from src.core.security.secure_encodings import SecureEmbedding

def create_sample_embeddings(num_samples=5, embedding_dim=512):
    """Create sample embeddings and metadata"""
    
    # Generate random embeddings
    embeddings = np.random.randn(num_samples, embedding_dim).astype(np.float32)
    
    # Create sample metadata
    metadata_list = []
    for i in range(num_samples):
        metadata = {
            "id": f"sample_{i}",
            "timestamp": datetime.now().isoformat(),
            "source": "test_data",
            "type": "text_embedding",
            "attributes": {
                "length": embedding_dim,
                "model": "test_model",
                "version": "1.0"
            }
        }
        metadata_list.append(metadata)
    
    return embeddings, metadata_list

def main():
    print("Starting secure embedding tests...")
    
    # Create sample data
    num_samples = 5
    embedding_dim = 512
    embeddings, metadata_list = create_sample_embeddings(num_samples, embedding_dim)
    
    print(f"\nCreated {num_samples} sample embeddings with dimension {embedding_dim}")
    print(f"Sample embedding shape: {embeddings[0].shape}")
    
    # Initialize secure embedding with a test key
    master_key = "test_secure_key_2024"
    secure_embed = SecureEmbedding(master_key)
    
    # Test encryption
    print("\nTesting embedding encryption...")
    try:
        encrypted_data = []
        secure_metadata = []
        
        for i in range(num_samples):
            enc_data, enc_meta = secure_embed.encrypt_embedding(
                embeddings[i],
                metadata_list[i],
                protection_level='high'
            )
            encrypted_data.append(enc_data)
            secure_metadata.append(enc_meta)
            
        print(f"Successfully encrypted {len(encrypted_data)} embeddings")
        
        # Print sample metadata
        print("\nSample secure metadata:")
        for key, value in secure_metadata[0].items():
            if key != 'access_token':  # Don't print the full token
                print(f"  {key}: {value}")
    
    except Exception as e:
        print(f"Error during encryption: {e}")
        raise
    
    # Test decryption
    print("\nTesting embedding decryption...")
    try:
        decrypted_embeddings = []
        
        for i in range(num_samples):
            dec_embedding = secure_embed.decrypt_embedding(
                encrypted_data[i],
                secure_metadata[i],
                secure_metadata[i]['access_token']
            )
            decrypted_embeddings.append(dec_embedding)
            
        print(f"Successfully decrypted {len(decrypted_embeddings)} embeddings")
        
        # Verify decryption accuracy
        print("\nVerifying decryption accuracy...")
        for i in range(num_samples):
            diff = np.abs(embeddings[i] - decrypted_embeddings[i])
            print(f"\nSample {i} statistics:")
            print(f"  Mean absolute difference: {diff.mean():.6f}")
            print(f"  Max absolute difference: {diff.max():.6f}")
            print(f"  Standard deviation: {diff.std():.6f}")
    
    except Exception as e:
        print(f"Error during decryption: {e}")
        raise
    
    # Test similarity search
    print("\nTesting secure similarity search...")
    try:
        # Create query embedding
        query_embedding = np.random.randn(embedding_dim).astype(np.float32)
        
        # Encrypt query
        enc_query, query_meta = secure_embed.encrypt_embedding(
            query_embedding,
            {"type": "query", "timestamp": datetime.now().isoformat()},
            protection_level='high'
        )
        
        # Perform secure search
        results = secure_embed.secure_similarity_search(
            enc_query,
            query_meta,
            encrypted_data,
            secure_metadata,
            k=3
        )
        
        print(f"\nFound {len(results)} similar embeddings")
        print("\nTop matches:")
        for i, (idx, score) in enumerate(results):
            print(f"  Match {i+1}:")
            print(f"    ID: {metadata_list[idx]['id']}")
            print(f"    Similarity score: {score:.4f}")
    
    except Exception as e:
        print(f"Error during similarity search: {e}")
        raise
    
    # Test batch operations
    print("\nTesting batch operations...")
    try:
        # Batch encryption
        batch_encrypted, batch_metadata = secure_embed.encrypt_batch(
            embeddings,
            metadata_list,
            protection_level='high'
        )
        print(f"Successfully encrypted batch of {len(batch_encrypted)} embeddings")
        
        # Batch decryption
        batch_decrypted = secure_embed.decrypt_batch(
            batch_encrypted,
            batch_metadata,
            [meta['access_token'] for meta in batch_metadata]
        )
        print(f"Successfully decrypted batch of {len(batch_decrypted)} embeddings")
        
        # Verify batch accuracy
        batch_diff = np.abs(embeddings - np.array(batch_decrypted))
        print("\nBatch processing accuracy:")
        print(f"  Mean absolute difference: {batch_diff.mean():.6f}")
        print(f"  Max absolute difference: {batch_diff.max():.6f}")
        print(f"  Standard deviation: {batch_diff.std():.6f}")
    
    except Exception as e:
        print(f"Error during batch operations: {e}")
        raise
    
    print("\nAll tests completed successfully!")


ModuleNotFoundError: No module named 'src.core.security.secure_encodings'

In [ ]:
main()